## Initialising ChipWhisperer Lite 

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb" 

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 13439905                  to 40024323                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538471                 
scope.clock.adc_rate                     changed from 0.0                       to 29538471.0        

In [3]:
def reboot_flush():
    reset_target(scope)
    #Flush garbage too
    target.flush()

## verifying password check output

In [6]:
#Do glitch loop
pw = bytearray([0x00]*5)
target.simpleserial_write(0x01, pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)
#print(bytearray(val['full_response'].encode('latin-1')))

{'valid': True, 'payload': CWbytearray(b'00'), 'full_response': CWbytearray(b'00 72 01 00 99 00'), 'rv': bytearray(b'\x00')}


In [7]:
#Do glitch loop
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68]) # correct password ASCII representation
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)
print(pw)

{'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}
bytearray(b'touch')


## Set up glitch

In [8]:
scope.cglitch_setup()

scope.clock.adc_freq                     changed from 29538471                  to 29538459                 
scope.clock.adc_rate                     changed from 29538471.0                to 29538459.0               


In [9]:
#Check if hs2 changed from clkgen to glitch, if not run the cell below
print(scope.io.hs2)

glitch


In [10]:
scope.io.hs2 = "glitch"
time.sleep(0.2)
reboot_flush()

## Set up glitch statistics and plot

In [11]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [12]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="ext_offset", y_index="offset")

Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [ext_offset,offset]
      .Points.II :Points   [ext_offset,offset]

## Glitch loop

In [13]:
def recover_from_hang():
    # make clock stable first (no glitch)
    scope.io.hs2 = "clkgen"
    time.sleep(0.05)

    reboot_flush()   # your improved NRST-based reboot_flush

    # go back to glitch output for the sweep
    scope.io.hs2 = "glitch"
    time.sleep(0.05)

In [14]:
from tqdm.notebook import tqdm
import re
import struct
import time

sample_size = 1


gc.set_global_step(1)
if PLATFORM == "CWLITEXMEGA":
        gc.set_range("width", 0, 10)
        gc.set_range("offset", -10,-10 )
        gc.set_range("ext_offset", 1, 100  )
        gc.set_step("ext_offset", 1)
        gc.set_step("width", 0.5)
        gc.set_step("offset", 0.5)
        #gc.set_global_step(0.2) 
        sample_size = 20

#gc.set_step("ext_offset", 1)
scope.glitch.repeat = 1
reboot_flush()
broken = False
scope.adc.timeout = 1

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    for i in range(sample_size):
        if scope.adc.state:
            # can detect crash here (fast) before timing out (slow)
            print("Trigger still high!")
            gc.add("reset")
            recover_from_hang()
            time.sleep(0.05)
            continue
            #Device is slow to boot?
            reboot_flush()

        scope.arm()
        target.simpleserial_write(0x01, bytes([0]*5))

        ret = scope.capture()

        if ret:
            print('Timeout - no trigger')
            gc.add("reset")

            #Device is slow to boot?
            reboot_flush()

        else:
            val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
            if val['valid'] is False:
                gc.add("reset")
            else:

                if val['payload'] == bytearray([1]): #for loop check
                    #broken = True
                    gc.add("success")
                    print(val['payload'])
                    print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                    print("✅", end="")
                    #break
                else:
                    gc.add("normal")
    if broken:
        break

(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration for width = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:790) Partial reconfiguration f

Trigger still high!


(ChipWhisperer Target WARNING|File SimpleSerial2.py:506) Read timed out: 
(ChipWhisperer Target ERROR|File SimpleSerial2.py:287) Device did not ack
(ChipWhisperer Target WARNING|File SimpleSerial2.py:381) Unexpected start to command 0
(ChipWhisperer Target WARNING|File SimpleSerial2.py:421) Invalid CRC. Expected 181 got 62
(ChipWhisperer Target WARNING|File SimpleSerial2.py:381) Unexpected start to command 0
(ChipWhisperer Target WARNING|File SimpleSerial2.py:421) Invalid CRC. Expected 181 got 62


## Results of most effective parameters

In [15]:
print(scope.adc.state)

False


In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="success_rate")
results

In [ ]:
results = gc.calc(ignore_params=["ext_offset"], sort="success_rate")
results

## Check for most reset rates

In [ ]:
results = gc.calc(ignore_params=["width", "offset"], sort="reset_rate")
results

In [ ]:
results = gc.calc(ignore_params=["ext_offset"], sort="reset_rate")
results